# Zemli: train the tomato disease model (free GPU)

Before running: **Runtime → Change runtime type → T4 GPU**. Then run each cell from top to bottom.

This trains on PlantVillage tomato images. These are lab photos on plain backgrounds, so the accuracy here will be much higher than in the field. Treat this as a starter model, then retrain with verified pilot-farm photos (see `import_verified.py`).

In [ ]:
!nvidia-smi -L
!pip -q install onnx

## 1. Upload `zemli-prototype.zip`
A file picker appears below. Choose the zip you downloaded.

In [ ]:
from google.colab import files
import glob, os, subprocess
up = files.upload()
# works whatever the file is called, e.g. "zemli-prototype (1).zip"
name = list(up)[0] if up else sorted(glob.glob("zemli-prototype*.zip"), key=os.path.getmtime)[-1]
subprocess.run(["unzip", "-q", "-o", name], check=True)
print("unzipped:", name)

## 2. Download PlantVillage tomato images (about 18,000 photos, a few minutes)

In [ ]:
FOLDERS = {
    "Tomato___healthy": "healthy",
    "Tomato___Early_blight": "early_blight",
    "Tomato___Late_blight": "late_blight",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "leaf_curl_virus",
    "Tomato___Septoria_leaf_spot": "septoria_leaf_spot",
    "Tomato___Leaf_Mold": "leaf_mold",
    "Tomato___Spider_mites Two-spotted_spider_mite": "spider_mites",
}
import subprocess
subprocess.run("git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git pv", shell=True, check=True)
subprocess.run(["git", "-C", "pv", "sparse-checkout", "set", "--no-cone"] + [f"raw/color/{f}/" for f in FOLDERS], check=True)
!ls pv/raw/color

## 3. Split into train / val / test (80 / 10 / 10, fixed seed)

In [ ]:
import random, shutil
from pathlib import Path
random.seed(42)
src, dst = Path("pv/raw/color"), Path("data/tomato")
shutil.rmtree(dst, ignore_errors=True)
for folder, label in FOLDERS.items():
    imgs = sorted((src / folder).glob("*.*"))
    random.shuffle(imgs)
    n = len(imgs); a, b = int(n * 0.8), int(n * 0.9)
    for split, part in [("train", imgs[:a]), ("val", imgs[a:b]), ("test_public", imgs[b:])]:
        d = dst / split / label; d.mkdir(parents=True, exist_ok=True)
        for p in part: shutil.copy(p, d / p.name)
    print(f"{label:20s} {n:5d} images")

## 4. Train (about 15-25 minutes on a T4)

In [ ]:
!python zemli/training/train.py --crop tomato --data data/tomato --epochs 8 --batch 64 --workers 2 --out out

## 5. Check the results
PlantVillage scores are always high. The number that matters later is `test_field`, once you add pilot photos.

In [ ]:
import json
m = json.load(open("out/tomato/metrics.json"))
for split in ["val", "test_public", "test_field"]:
    if split in m: print(f"{split:12s} accuracy {m[split]['accuracy']:.4f}  macro-F1 {m[split]['macro_f1']:.4f}")
print()
for c, v in m["test_public"]["per_class"].items(): print(f"{c:20s} F1 {v['f1']:.3f}  ({v['support']} images)")

## 6. Download the model
Unzip it into `backend/models/` on your laptop so you get `backend/models/tomato/model.onnx` and `labels.txt`. Then restart uvicorn.

In [ ]:
!cd out && zip -q -r ../tomato_model.zip tomato/model.onnx tomato/labels.txt tomato/metrics.json
files.download("tomato_model.zip")

## 7. (Optional) Convert to TFLite for the Flutter app
Only needed for the offline Flutter app. The web field app uses the server model and doesn't need this. If this cell fails because of package versions, skip it for now.

In [ ]:
!pip -q install onnx2tf tensorflow onnxsim onnx_graphsurgeon sng4onnx psutil tf_keras
!onnx2tf -i out/tomato/model.onnx -o out/tomato/tflite -b 1
!cp out/tomato/labels.txt out/tomato/tflite/tomato_labels.txt && cp out/tomato/tflite/model_float16.tflite out/tomato/tflite/tomato.tflite
files.download("out/tomato/tflite/tomato.tflite"); files.download("out/tomato/tflite/tomato_labels.txt")